To return plots generated by `opti_multi_pump_clean.py`, switch the file names found in the folder 'Output' and run every cell in this notebook. The plots should pop out and be interactive.

In [2]:
# Outputs
file_path = r'..\src\Output\1_retrofit_electric_20260130_113434_data_output.xlsx'
pkl_file_path = r'..\src\Output\1_retrofit_electric_20260130_113434_pipe_data_output.pkl'

# Data (Gogma)
data_file = r'..\..\DO_NOT_DISTRIBUTE\DO_NOT_DISTRIBUTE_DATA_GOGMA.xlsx'

In [3]:
import pandas as pd
import numpy as np
import pickle
import networkx as nx
%matplotlib tk
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from geopy.distance import great_circle
from matplotlib.lines import Line2D

In [4]:
def recover_data(file_path, pkl_file_path):
    data_dict = pd.read_excel(file_path, sheet_name=None)
    all_positions = []
    all_values = []
    all_indices = []
    all_X = []
    max_nb_fountains = len(data_dict) - 1 
    initial_impact = data_dict['data_summary']['initial_impact'][0]
    
    counter = 0
    for sheet_name, df in data_dict.items():
        if sheet_name != 'data_summary':
            all_positions.append(df.iloc[:,counter:-2].to_numpy())
            all_values.append(df.iloc[:,-2:].to_numpy())
            all_indices.append(df.iloc[:,:counter].to_numpy())
            all_X.append(df.iloc[:,:-2].to_numpy())
        counter += 1

    pipe_data = pickle.load(open(pkl_file_path,'rb'))
    return all_positions, all_values, all_indices, all_X, max_nb_fountains, initial_impact, pipe_data

In [5]:
def read_data_gogma(file_path):
    data_households, data_sources = pd.read_excel(
        file_path,
        sheet_name=['Position surveyed household (hh','Position water sources'],
        header=None).values()
    data_sources = data_sources.iloc[:,:5]
    data_sources.columns = ['ID','Name','Lon','Lat','Altitude']
    data_sources['Type'] = data_sources['Name'].str[1]
    data_sources = data_sources[(data_sources['Type']=='B') | (data_sources['Type']=='W')].drop('Name',axis=1)
    data_sources['Type'] = data_sources['Type'].replace('B','Borehole')
    data_sources['Type'] = data_sources['Type'].replace('W','Open Well')
    
    data_households.columns = ['ID','Lat','Lon','Altitude']
    data_household_capita = pd.read_excel(file_path,sheet_name='Choice source before install',usecols='A,F,G').iloc[1:]
    data_household_capita.columns = ['ID','Reading','Nb capita']
    data_household_capita = data_household_capita.sort_values(by='Reading',ascending=True)
    data_household_capita = data_household_capita.drop_duplicates(subset='ID').sort_values(by='ID').iloc[:-1].drop('Reading',axis=1)
    
    data_households['Type'] = 'Household'
    data_households = pd.merge(data_households, data_household_capita, on='ID', how='inner')    
    
    data = pd.concat([data_households,data_sources],ignore_index=True)
    households = data_households
    pumps = data_sources[data_sources['Type']=='Borehole']
    open_wells = data_sources[data_sources['Type']=='Open Well']
    
    
    return data,households,pumps,open_wells

In [ ]:
class InteractiveParetoPlot:
    def __init__(self, all_positions, all_result_vals, all_indices, 
                 households, pos_pumps, grid_x, grid_y, grid_z, initial_impact, pipe_data,impactfn,
                 max_nb_fountains, all_X=None):
        # Store data
        self.all_positions = all_positions
        self.all_result_vals = all_result_vals
        self.all_indices = all_indices
        self.households = households
        self.pos_pumps = pos_pumps
        self.grid_x = grid_x
        self.grid_y = grid_y
        self.grid_z = grid_z
        self.initial_impact = initial_impact
        self.pipe_data = pipe_data
        self.impactfn = impactfn
        self.max_nb_fountains = max_nb_fountains
        self.all_X = all_X if all_X is not None else []
        
        # Create separate figures for each plot
        self.fig1, self.ax1 = plt.subplots(figsize=(10, 6))  # Pareto front
        self.fig1.suptitle('Pareto Front')
        
        self.fig2, self.ax2 = plt.subplots(figsize=(12, 10))  # Geographical map
        self.fig2.suptitle('Geographical Map')
        
        self.fig3, self.ax3 = plt.subplots(figsize=(10, 8))  # Network graph
        self.fig3.suptitle('Network Topology')
        
        # Store references to plot elements for highlighting
        self.pareto_scatter = None
        self.pump_scatters = []
        self.pump_lines = []
        self.highlighted_pumps = []
        self.highlighted_lines = []
        self.data_box = None  # Single data box that gets updated
        self.n_tested = np.arange(1,self.max_nb_fountains+1,dtype=int)
        
        # Current hover index
        self.current_hover_idx = None
        
        self.setup_plots()
        self.connect_events()
    
    def setup_plots(self):
        """Set up both plots"""
        # === PARETO PLOT (ax1) ===
        FILLED_MARKERS = list(Line2D.filled_markers)
        n_shape = {key: FILLED_MARKERS[i % len(FILLED_MARKERS)] for i, key in enumerate(self.n_tested)}
        
        # Store all scatter plots for hover detection
        self.pareto_scatters = []
        if self.impactfn == "impact2":
            for n, k in enumerate(self.n_tested):
                scatter = self.ax1.scatter(
                    -1*self.all_result_vals[n][:, 0], 
                    self.all_result_vals[n][:, 1],
                    c=-1*self.all_result_vals[n][:, 0],
                    cmap='viridis',
                    vmin=-100,
                    vmax=0,
                    edgecolor='k',
                    s=20,
                    label=f'{k} new fountains',
                    marker=n_shape.get(k,'s'),
                )
                self.pareto_scatters.append(scatter)
            
            self.ax1.set_xlabel('% of people within 30 mins of nearest improved water source', fontsize=12)
            self.ax1.set_xlim(50,105)
        else:
            for n, k in enumerate(self.n_tested):
                scatter = self.ax1.scatter(
                    self.all_result_vals[n][:, 0]-self.initial_impact, 
                    self.all_result_vals[n][:, 1],
                    c=self.all_result_vals[n][:, 0]-self.initial_impact,
                    cmap='viridis',
                    vmin=-100,
                    vmax=0,
                    edgecolor='k',
                    s=20,
                    label=f'{k} new fountains',
                    marker=n_shape.get(k,'s'),
                )
                self.pareto_scatters.append(scatter)
        
            self.ax1.set_xlabel('Impact (thousand person-meters saved)', fontsize=12)
            
        self.ax1.set_ylabel('Cost (€)', fontsize=12)
        self.ax1.set_title('Pareto Front', fontsize=12)
        self.ax1.legend()
        self.ax1.grid(True)
        
        # === GEOGRAPHICAL PLOT (ax2) ===
        contour = self.ax2.contourf(self.grid_x, self.grid_y, self.grid_z, levels=5, cmap='terrain')
        
        # Plot households (black dots)
        self.ax2.scatter(self.households['Lon'], self.households['Lat'], color='black', label='Households')
        
        # Plot new fountains and store references
        pump_idx = 0  # Index to track pumps across all configurations
        
        for n, k in enumerate(self.n_tested):
            if self.impactfn == "impact2":
                for i in range(k):
                    pos_pumps_new_plot = self.all_positions[n][:,2*i:2*(i+1)]
                    sc = self.ax2.scatter(
                        pos_pumps_new_plot[:, 0], pos_pumps_new_plot[:, 1],
                        c=-1*self.all_result_vals[n][:, 0],
                        cmap='viridis', 
                        edgecolor='k',
                        vmin=50,
                        vmax=100,
                        s=60,
                        label='New fountains' if pump_idx == 0 else "",
                        marker=n_shape.get(k,'s'),
                        alpha=0.5
                    )
                    self.pump_scatters.append(sc)
                    
                    # Store lines for each solution in this pump configuration
                    lines_for_config = []
                    for j in range(len(pos_pumps_new_plot)):  # For each solution
                        solution_lines = []
                        if j < len(self.all_indices[n]) and i < self.all_indices[n].shape[1]:
                            if self.all_indices[n][j,i] < len(self.pos_pumps):
                                x = int(self.all_indices[n][j,i])
                                parent_coord = self.pos_pumps[x]
                            else:
                                x = int(self.all_indices[n][j,i] - len(self.pos_pumps))
                                if x < len(pos_pumps_new_plot):
                                    parent_coord = self.all_positions[n][j,2*x:2*(x+1)]
                                else:
                                    continue
                            
                            points_to_plot = np.array([pos_pumps_new_plot[j], parent_coord])
                            line, = self.ax2.plot(
                                points_to_plot[:, 0], points_to_plot[:, 1],
                                c='black',
                                alpha=0,
                                linewidth=1
                            )
                            solution_lines.append(line)
                        lines_for_config.append(solution_lines)
                    
                    self.pump_lines.append(lines_for_config)
                    pump_idx += 1
            else:
                for i in range(k):
                    pos_pumps_new_plot = self.all_positions[n][:,2*i:2*(i+1)]
                    sc = self.ax2.scatter(
                        pos_pumps_new_plot[:, 0], pos_pumps_new_plot[:, 1],
                        c=self.all_result_vals[n][:, 0]-self.initial_impact,
                        cmap='viridis', 
                        edgecolor='k',
                        vmin=-100,
                        vmax=0,
                        s=60,
                        label='New fountains' if pump_idx == 0 else "",
                        marker=n_shape.get(k,'s'),
                        alpha=0.5
                    )
                    self.pump_scatters.append(sc)
                    
                    # Store lines for each solution in this pump configuration
                    lines_for_config = []
                    for j in range(len(pos_pumps_new_plot)):  # For each solution
                        solution_lines = []
                        if j < len(self.all_indices[n]) and i < self.all_indices[n].shape[1]:
                            if self.all_indices[n][j,i] < len(self.pos_pumps):
                                x = int(self.all_indices[n][j,i])
                                parent_coord = self.pos_pumps[x]
                            else:
                                x = int(self.all_indices[n][j,i] - len(self.pos_pumps))
                                if x < len(self.all_positions[n]):
                                    parent_coord = self.all_positions[n][j,2*x:2*(x+1)]
                                else:
                                    continue
                            
                            points_to_plot = np.array([pos_pumps_new_plot[j], parent_coord])
                            line, = self.ax2.plot(
                                points_to_plot[:, 0], points_to_plot[:, 1],
                                c='black',
                                alpha=0,
                                linewidth=1
                            )
                            solution_lines.append(line)
                        lines_for_config.append(solution_lines)
                    
                    self.pump_lines.append(lines_for_config)
                    pump_idx += 1
        
        # Plot previous pumps (red)
        self.ax2.scatter(self.pos_pumps[:, 0], self.pos_pumps[:, 1], 
                        color='red', label='Previous Pumps', edgecolors='k', s=90)
        
        # # Add colorbar
        # if len(self.pump_scatters) > 0:
        #     cbar = self.fig.colorbar(self.pareto_scatters[-1], ax=self.ax2)
        #     if self.impactfn=="impact2":
        #         cbar.set_label('% of people within 30 mins')
        #     else:
        #         cbar.set_label('Impact (thousand person-meters)')
        
        # Labels and title
        self.ax2.set_xlabel('Longitude')
        self.ax2.set_ylabel('Latitude')
        self.ax2.set_title('Households and Pumps')
        self.ax2.legend()
        self.ax2.grid(True)
        
        self.fig1.tight_layout()
        self.fig2.tight_layout()
        self.fig3.tight_layout()
    
    def connect_events(self):
        """Connect mouse events to the Pareto plot figure"""
        self.fig1.canvas.mpl_connect('motion_notify_event', self.on_hover)
    
    def on_hover(self, event):
        """Handle hover events over the Pareto plot"""
        if event.inaxes == self.ax1:
            # Check each scatter plot for hover
            hover_idx = None
            cumulative_solutions = 0
            
            for n, scatter in enumerate(self.pareto_scatters):
                cont, ind = scatter.contains(event)
                if cont:
                    # Get the index within this scatter plot
                    local_idx = ind["ind"][0]
                    # Convert to global index
                    hover_idx = cumulative_solutions + local_idx
                    break
                cumulative_solutions += len(self.all_result_vals[n])
            
            if hover_idx is not None:
                if hover_idx != self.current_hover_idx:
                    self.highlight_solution(hover_idx)
                    self.current_hover_idx = hover_idx
            else:
                # Clear highlighting if not hovering over any point
                if self.current_hover_idx is not None:
                    self.clear_highlighting()
                    self.current_hover_idx = None
    
    def get_pipe_data_text(self, config_idx, solution_in_config, solution_X=None):
        """Get formatted pipe data text for display"""
        try:
            # Access pipe_data by X key (solution_X required for dict-based storage)
            if config_idx < len(self.pipe_data):
                config_data = self.pipe_data[config_idx]
                
                # Dict-based storage: use X as key
                if isinstance(config_data, dict):
                    if solution_X is not None:
                        solution_key = tuple(solution_X.flatten())
                        for key in config_data.keys():
                            if np.allclose(solution_key, key):
                                solution_key = key
                                return config_data[solution_key]
                # Legacy list-based storage
                elif isinstance(config_data, list):
                    if solution_in_config < len(config_data):
                        return config_data[solution_in_config]
            
            return "No data available"
        except (IndexError, KeyError, TypeError) as e:
            print(f"Error retrieving pipe data: {e}")
            return "No data available"
    
    def highlight_solution(self, solution_idx):
        """Highlight the corresponding solution in the geographical plot"""
        self.clear_highlighting()
        
        # Find which configuration this solution belongs to
        cumulative_solutions = 0
        config_idx = None
        solution_in_config = None
        solution_X = None
        
        for n, k in enumerate(self.n_tested):
            num_solutions = len(self.all_result_vals[n])
            if cumulative_solutions <= solution_idx < cumulative_solutions + num_solutions:
                config_idx = n
                solution_in_config = solution_idx - cumulative_solutions
                solution_X = self.all_X[n][solution_in_config]
                break
            cumulative_solutions += num_solutions
        
        if config_idx is not None and solution_in_config is not None and solution_X is not None:
            # Highlight pumps for this specific solution
            pump_config_idx = 0
            for n in range(config_idx + 1):
                k = self.n_tested[n]
                if n == config_idx:
                    # This is our target configuration
                    for i in range(k):
                        if pump_config_idx < len(self.pump_scatters):
                            # Get the specific solution's pump positions
                            pos_pumps_new_plot = self.all_positions[n][:,2*i:2*(i+1)]
                            highlighted_scatter = self.ax2.scatter(
                                pos_pumps_new_plot[solution_in_config:solution_in_config+1, 0],
                                pos_pumps_new_plot[solution_in_config:solution_in_config+1, 1],
                                c='yellow',
                                s=150,
                                marker='*',
                                edgecolor='red',
                                linewidth=2,
                                zorder=10
                            )
                            self.highlighted_pumps.append(highlighted_scatter)
                            
                            # Highlight only the lines for this specific solution
                            if pump_config_idx < len(self.pump_lines):
                                if solution_in_config < len(self.pump_lines[pump_config_idx]):
                                    solution_lines = self.pump_lines[pump_config_idx][solution_in_config]
                                    for line in solution_lines:
                                        line.set_color('red')
                                        line.set_linewidth(3)
                                        line.set_alpha(1.0)
                                        self.highlighted_lines.append(line)
                        
                        pump_config_idx += 1
                else:
                    pump_config_idx += k
            
            # Show data box
            self.show_data_box(config_idx, solution_in_config, solution_X)
            
            # Refresh all plots
            self.fig2.canvas.draw_idle()
            self.fig3.canvas.draw_idle()
            
    def get_node_families_dfs_by_level(self, G):
        """Get nodes grouped by family and level using DFS from root nodes."""
        # Find all root nodes
        root_nodes = [node for node in G.nodes() if G.in_degree(node) == 0]
        
        families = []
        
        for root in root_nodes:
            family_dict = {}
            visited = set()
            
            def dfs(node, level):
                if node not in visited:
                    visited.add(node)
                    # Add node to the level dictionary
                    if level not in family_dict:
                        family_dict[level] = []
                    family_dict[level].append(node)
                    
                    # Recursively visit successors at the next level
                    for successor in G.successors(node):
                        dfs(successor, level + 1)
            
            dfs(root, 0)
            if family_dict:
                families.append(family_dict)
        
        return families
    
    def show_data_box(self, config_idx, solution_in_config, solution_X=None):
        """Show data box with pipe information"""
        # Get the formatted data text
        data_graph = self.get_pipe_data_text(config_idx, solution_in_config, solution_X)
        families = self.get_node_families_dfs_by_level(data_graph)
        data_string = ""
        for i, family in enumerate(families):
            for level, nodes in family.items():
                for node in nodes:
                    node = str(node)
                    if node.startswith('N'):
                        node_data = data_graph.nodes[node]
                        parent = list(data_graph.predecessors(node))[0]
                        edge_data = data_graph.get_edge_data(parent, node)
                        
                        
                        lon, lat = node_data['coords']
                        fountain_head = node_data['head']
                        diameter = edge_data['diameter']
                        pump_power = edge_data['pump_power']
                        pump_head = edge_data['pump_head']
                        flow_rate = edge_data['flow_rate']*1000  # convert to L/s
                        
                        data_string += f"Lon: {lon:.3f}, Lat: {lat:.3f}, Fountain head: {fountain_head:.3f}m, \n Diameter: {diameter:.3f}m, Power: {pump_power:.3f}W, Head Change: {pump_head:.3f}m, Flow Rate: {flow_rate:.3f}L/s; \n"
        
        
        # Create box properties
        props = dict(boxstyle='round', facecolor='lightblue', alpha=0.8, edgecolor='black')
        # print(solution_X)
        
        # Add solution info to the text
        k = self.n_tested[config_idx]
        header = f"Solution {self.current_hover_idx}\n{k} Fountain{'s' if k > 1 else ''}\n---\n"
        full_text = header + data_string
        
        # Place text box in upper left of geographical plot
        self.data_box = self.ax2.text(
            0.02, 0.98, full_text, 
            transform=self.ax2.transAxes, 
            fontsize=8,
            verticalalignment='top',
            horizontalalignment='left',
            bbox=props,
            zorder=15
        )
        
        # Draw the network graph
        self.draw_network_graph(config_idx, solution_in_config, solution_X)
    
    def hide_data_box(self):
        """Hide the data box"""
        if self.data_box is not None:
            self.data_box.remove()
            self.data_box = None
    
    def clear_highlighting(self):
        """Clear all highlighting"""
        # Remove highlighted pumps
        for scatter in self.highlighted_pumps:
            scatter.remove()
        self.highlighted_pumps.clear()
        
        # Reset line properties
        for line in self.highlighted_lines:
            line.set_color('black')
            line.set_linewidth(1)
            line.set_alpha(0)
        self.highlighted_lines.clear()
        
        # Hide data box
        self.hide_data_box()
        
        # Clear graph plot
        self.ax3.clear()
        
        # Refresh all plots
        self.fig2.canvas.draw_idle()
        self.fig3.canvas.draw_idle()
    
    def draw_network_graph(self, config_idx, solution_in_config, solution_X=None):
        """Draw the network graph on ax3"""
        # Clear previous graph
        self.ax3.clear()
        
        # Get the graph
        graph = self.get_pipe_data_text(config_idx, solution_in_config, solution_X)
        
        # 1. Start with your geographic positions
        initial_pos = {node: np.array(graph.nodes[node].get('coords', (0, 0))) for node in graph.nodes()}

        # 2. Use spring_layout to 'nudge' nodes apart
        # k=0.1 to 0.5 controls the distance between nodes
        pos = nx.spring_layout(
            graph, 
            # pos=initial_pos, 
            fixed=None,      # Allow all nodes to move slightly
            k=0.3,           # Increase this if labels still overlap
            iterations=50
            )
        
        if graph is None or len(graph.nodes()) == 0:
            self.ax3.text(0.5, 0.5, "No graph data", ha='center', va='center')
            self.ax3.set_xlim(0, 1)
            self.ax3.set_ylim(0, 1)
            return
        
        # Draw nodes
        node_colors = []
        for node in graph.nodes():
            node = str(node)
            if node.startswith('F'):
                node_colors.append('red')  # Fixed pumps in red
            else:
                node_colors.append('lightblue')  # New fountains in light blue
        
        nx.draw_networkx_nodes(
            graph, pos, ax=self.ax3,
            node_color=node_colors,
            node_size=500,
            alpha=0.9,
            linewidths=2
        )
        
        
        # Draw edges with straight arrows
        nx.draw_networkx_edges(
            graph, pos, ax=self.ax3,
            edge_color='gray',
            arrows=True,
            arrowsize=20,
            arrowstyle='->',
            connectionstyle="arc3,rad=0"
        )
        # Create node labels with all node data
        node_labels = {}
        for node in graph.nodes():
            node_data = graph.nodes[node]
            head = node_data.get('head', 0)
            fixed_cost = node_data.get('fixed_cost', 0)
            coords = node_data.get('coords', (0, 0))
            node_labels[node] = f"{node}\nHead:{head:.1f}m\nCost:€{fixed_cost:.0f}\nLon:{coords[0]:.4f}\nLat:{coords[1]:.4f}"
        
        nx.draw_networkx_labels(
            graph, pos, node_labels, ax=self.ax3,
            font_size=7,
        )
        
        # Create edge labels with all edge data
        edge_labels = {}
        for u, v, data in graph.edges(data=True):
            diameter = data.get('diameter', 'N/A')
            flow_rate = data.get('flow_rate', 'N/A')
            length = data.get('weight', 'N/A')
            pump_power = data.get('pump_power', 'N/A')
            pipe_cost = data.get('pipe_cost', 'N/A')
            
            if isinstance(flow_rate, (int, float)):
                flow_rate = f"{flow_rate*1000:.2f}L/s"
            if isinstance(diameter, (int, float)):
                diameter = f"{diameter:.4f}m"
            if isinstance(length, (int, float)):
                length = f"{length:.1f}m"
            if isinstance(pump_power, (int, float)):
                pump_power = f"{pump_power:.1f}W"
            if isinstance(pipe_cost, (int, float)):
                pipe_cost = f"€{pipe_cost:.2f}"
            
            edge_labels[(u, v)] = f"∅:{diameter}\nQ:{flow_rate}\nL:{length}\nPower:{pump_power}\nCost:{pipe_cost}"
        
        nx.draw_networkx_edge_labels(
            graph, pos, edge_labels, ax=self.ax3,
            font_size=6
        )
        
        
        self.ax3.set_title('Pipe Network Topology', fontsize=12, fontweight='bold')
        self.ax3.axis('off')
    
    def show(self):
        """Display the interactive plot"""
        plt.show()

In [10]:
# Import data and define arrays (may need to rewrite read function depending on input)
data,households,pumps,open_wells = read_data_gogma(data_file)
pos_households = households[['Lon','Lat']].to_numpy() 
nb_capita = households['Nb capita'].to_numpy()
pos_pumps = pumps[['Lon','Lat']].to_numpy()
bounds = np.array([
    [data['Lon'].min(), data['Lat'].min()],  # Min bounds
    [data['Lon'].max(), data['Lat'].max()],  # Max bounds
])

# Create gridpoints for relief plot
grid_x, grid_y = np.mgrid[data['Lon'].min():data['Lon'].max():200j, 
                        data['Lat'].min():data['Lat'].max():200j]
grid_z = griddata((data['Lon'], data['Lat']), data['Altitude'], (grid_x, grid_y), method='cubic')

In [11]:
all_positions, all_result_vals, all_indices, all_X, max_nb_fountains, initial_impact, pipe_data = recover_data(file_path, pkl_file_path)
plotter = InteractiveParetoPlot(all_positions, all_result_vals, all_indices, 
                      households, pos_pumps, grid_x, grid_y, grid_z, initial_impact, pipe_data, "impact",
                      max_nb_fountains, all_X)
plotter.show()

### Comparing two pareto fronts

In [31]:
# Outputs
file_path = [r'..\src\Output\1_retrofit_electric_20260130_113434_data_output.xlsx',
             r'..\src\Output\2_retrofit_electric_20260130_143400_data_output.xlsx',
             ]
pkl_file_path = [r'..\src\Output\1_retrofit_electric_20260130_113434_pipe_data_output.pkl',
                 r'..\src\Output\2_retrofit_electric_20260130_143400_pipe_data_output.pkl',
                 ]

In [32]:
result_vals = []
initial_imps = []
for excel,pkl_file in zip(file_path,pkl_file_path):
    all_positions, all_result_vals, all_indices, all_X, max_nb_fountains, initial_impact, pipe_data = recover_data(excel, pkl_file)
    result_vals.append(all_result_vals)
    initial_imps.append(initial_impact)

In [62]:
colours = ['green','red']
shapes = ['.','X','^']

In [65]:
fig,ax = plt.subplots(figsize =  (12,7))
for i,results in enumerate(result_vals):
    initial_impact_val = initial_imps[i]
    for j,solutions in enumerate(results):
        label = f'n_retrofit = {i+1}, n_fountains = {j}'
        ax.scatter(solutions[:,0]-initial_impact_val,solutions[:,1],label=label, marker=shapes[j],color=colours[i])

ax.set_xlabel("Impact (person-meters $*10^3$)")
ax.set_ylabel("Cost (€)")
ax.legend()
ax.grid()
fig.savefig("pareto_comparison.svg")
plt.show()